# 01 — RAG ingestion

Jednorazowy pipeline do dodawania nowego dokumentu miejskiego do bazy Pismaka.

**Ważne:** `zarządzenie` jest przechowywane jako `legal_form`, a nie jako
poziom tej samej hierarchii co strategia/polityka/standard.

In [ ]:
from pathlib import Path
import os, sys, json

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "docker-compose.yaml").exists()
        and (candidate / "src").is_dir()
    ),
    Path.cwd(),
)
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from Zrodlowyszukiwator.rag_ingestion.catalog import load_catalog
from Zrodlowyszukiwator.rag_ingestion.embedder import OllamaEmbedder
from Zrodlowyszukiwator.rag_ingestion.pipeline import ingest_pdf
from Zrodlowyszukiwator.rag_ingestion.repository import PgVectorRepository
from Zrodlowyszukiwator.rag_ingestion.pdf_extractor import extract_pdf_pages
from Zrodlowyszukiwator.rag_ingestion.chunker import build_text_units, chunk_units

print("Gotowe.")

In [ ]:
# Hierarchia MERYTORYCZNA — nie jest to hierarchia prawna.
HIERARCHY = {
    "strategy": 100,
    "policy": 80,
    "program": 70,
    "plan": 70,
    "standard": 60,
    "procedure": 50,
    "other": 40,
}

HIERARCHY

In [ ]:
# Konfiguracja.
# Wskaż jeden plik dodawany w tej sesji.

POLICIES_DIR = PROJECT_ROOT / "src" / "Zrodlowyszukiwator" / "RAG" / "RAW"
CATALOG_PATH = PROJECT_ROOT / "src" / "Zrodlowyszukiwator" / "RAG" / "document_catalog.json"

PDF_PATH = POLICIES_DIR / "00_Strategia-rozwoju-Wroclawia-Wroclaw-2050.pdf"

print(CATALOG_PATH)
catalog = load_catalog(CATALOG_PATH)

if PDF_PATH.name not in catalog:
    raise KeyError(
        f"Brak metadanych dla {PDF_PATH.name!r} w {CATALOG_PATH}"
    )

metadata = catalog[PDF_PATH.name]
display(metadata)

In [ ]:
# Etap A — ekstrakcja PDF i kontrola jakości tekstu.

pages = extract_pdf_pages(PDF_PATH)

print("Liczba stron:", len(pages))
print("Strony z bardzo małą ilością tekstu:",
      [p.page_number for p in pages if p.low_text_quality])

print("\n--- przykład strony ---\n")
print(pages[min(2, len(pages)-1)].text[:2500])

In [ ]:
# Etap B — sekcje i chunking.

units = build_text_units(pages)

chunks = chunk_units(
    units,
    metadata,
    max_chars=3500,
    min_chars=250,
    overlap_units=1,
)

print("Text units:", len(units))
print("Chunks:", len(chunks))

for chunk in chunks[:3]:
    print("\n" + "="*80)
    print("chunk_id:", chunk.chunk_id)
    print("pages:", chunk.page_start, "-", chunk.page_end)
    print("section:", " > ".join(chunk.section_path))
    print(chunk.text[:1200])

In [ ]:
# Etap C — PostgreSQL + pgvector.

DATABASE_URL = os.environ["DATABASE_URL"]
EMBEDDING_DIM = int(os.getenv("RAG_EMBEDDING_DIM", "1024"))

repo = PgVectorRepository(
    DATABASE_URL,
    embedding_dim=EMBEDDING_DIM,
)
repo.ensure_schema()

print("Schema gotowy.")

In [ ]:
# Etap D — lokalne embeddingi przez Ollamę.

embedder = OllamaEmbedder(
    model=os.getenv("RAG_EMBEDDING_MODEL", "bge-m3"),
    base_url=os.getenv("OLLAMA_URL", "http://127.0.0.1:11434"),
    batch_size=16,
)

test_embedding = embedder.embed(["Test embeddingu dla dokumentów miejskich."])[0]
print("Wymiar embeddingu:", len(test_embedding))

if len(test_embedding) != EMBEDDING_DIM:
    raise ValueError(
        f"Model zwraca {len(test_embedding)} wymiarów, "
        f"a DB jest skonfigurowana na {EMBEDDING_DIM}."
    )

In [ ]:
# Etap E — pełny ingestion.
# Ponowne uruchomienie dla identycznego pliku nie tworzy duplikatów.

result = ingest_pdf(
    PDF_PATH,
    metadata,
    repository=repo,
    embedder=embedder,
    max_chars=3500,
    min_chars=250,
    overlap_units=1,
    force=False,
)

result

In [ ]:
# Kontrola bazy.

doc_count, chunk_count = repo.counts()

print("Dokumenty w bazie:", doc_count)
print("Chunki w bazie:", chunk_count)

## Dodawanie następnego dokumentu

1. Skopiuj PDF do katalogu `polityki miejskie/`.
2. Dopisz jego metadata do `examples/document_catalog.json`.
3. Zmień `PDF_PATH`.
4. Uruchom komórki od ekstrakcji do ingestion.

Po ustabilizowaniu formatu metadanych notebook można zastąpić CLI:

```cmd
python -m Zrodlowyszukiwacz.rag_ingestion.cli ^
  --file "polityki miejskie\nowy.pdf" ^
  --catalog examples\document_catalog.json
```